# Qwen3-8B 스터디 추천 LoRA 학습 (Unsloth)

- **태스크**: 사용자 주제/기술스택/일정 → 스터디 계획 JSON 생성
- **GPU**: SSAFY L40S 46GB
- **D106팀 - GPU Device 3**

14B OOM 시 대체용. 8B는 여유 있음.

## 1. GPU 설정 + 환경 확인

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import gc
import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"VRAM: {total_mem:.1f} GB")

## 2. 데이터 로드 + 토큰 길이 분석

In [ ]:
import json
import numpy as np
from transformers import AutoTokenizer

TRAIN_PATH = "data_collection/recommend_training_data_train.json"
VAL_PATH = "data_collection/recommend_training_data_val.json"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_raw = json.load(f)
with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_raw = json.load(f)

print(f"Train: {len(train_raw)}개")
print(f"Val: {len(val_raw)}개")
print(f"\n샘플 키: {list(train_raw[0].keys())}")
print(f"샘플 text 앞 300자:\n{train_raw[0]['text'][:300]}")

In [ ]:
# 토큰 길이 분석
tokenizer_check = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B", trust_remote_code=True)

lengths = [len(tokenizer_check.encode(d["text"])) for d in train_raw]

print(f"[토큰 길이 분포]")
print(f"  평균: {np.mean(lengths):.0f}")
print(f"  중앙값: {np.median(lengths):.0f}")
print(f"  P95: {np.percentile(lengths, 95):.0f}")
print(f"  P99: {np.percentile(lengths, 99):.0f}")
print(f"  최대: {max(lengths)}")
print(f"  최소: {min(lengths)}")

p99 = int(np.percentile(lengths, 99))
recommended = ((p99 // 64) + 1) * 64
print(f"\n추천 MAX_SEQ_LENGTH: {recommended} (P99={p99} 기준)")
print(f"1024 초과: {sum(1 for l in lengths if l > 1024)}개")
print(f"2048 초과: {sum(1 for l in lengths if l > 2048)}개")

del tokenizer_check
gc.collect()

## 3. 설정

In [ ]:
# ===== 모델 설정 =====
MODEL_NAME = "unsloth/Qwen3-8B"
USE_4BIT = True

# ⚠️ 위 토큰 길이 분석 결과에 따라 조정
MAX_SEQ_LENGTH = 2048  # 8B는 여유 있으니 넉넉하게

# ===== LoRA 설정 (8B는 여유) =====
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0

# ===== 학습 하이퍼파라미터 =====
BATCH_SIZE = 2          # 8B는 2 가능
GRAD_ACCUM = 4          # effective batch = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_RATIO = 0.1

OUTPUT_DIR = "./outputs/qwen3-8b-recommend"

print(f"모델: {MODEL_NAME}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"Epochs: {NUM_EPOCHS}")

## 4. 모델 로드 (Unsloth 4-bit)

In [ ]:
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()

print(f"모델 로딩: {MODEL_NAME}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=USE_4BIT,
)

print(f"모델 로드 완료!")
print(f"VRAM: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

## 5. LoRA 설정

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터: {trainable:,} ({100 * trainable / total:.2f}%)")
print(f"VRAM: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

## 6. [BEFORE] 파인튜닝 전 테스트

In [ ]:
TEST_INPUT = """스터디 주제: Spring Boot 학습
기술 스택: Java, Spring Boot, JPA, MySQL, Redis, Docker, Git
가용 일정: 화 20:00-22:00, 목 20:00-22:00
선호 기간: 4주

위 정보를 바탕으로 완성된 스터디 계획을 JSON으로 생성해주세요."""

SYSTEM_PROMPT = train_raw[0]["text"].split("<|im_start|>system\n")[1].split("<|im_end|>")[0]

def run_inference(model, tokenizer, user_input, system_prompt=SYSTEM_PROMPT):
    FastLanguageModel.for_inference(model)
    prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_input}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    start = response.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
    end = response.find("<|im_end|>", start)
    return response[start:end].strip() if end != -1 else response[start:].strip()

print("=" * 60)
print("[BEFORE] 파인튜닝 전")
print("=" * 60)
before_result = run_inference(model, tokenizer, TEST_INPUT)
print(before_result[:800])

## 7. 데이터셋 준비

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_raw)
val_dataset = Dataset.from_list(val_raw)

print(f"Train: {len(train_dataset)}개")
print(f"Val: {len(val_dataset)}개")

## 8. Trainer 설정 + 학습 실행

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

FastLanguageModel.for_training(model)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    optim="adamw_8bit",
    bf16=True,
    fp16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

print("Trainer 준비 완료")
print(f"VRAM: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

In [ ]:
gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("학습 시작!")
print("=" * 60)

train_result = trainer.train()

print(f"\n학습 완료!")
print(f"Steps: {train_result.global_step}")
print(f"Train Loss: {train_result.training_loss:.4f}")

## 9. 학습 곡선

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_loss = [(x["step"], x["loss"]) for x in logs if "loss" in x]
eval_loss = [(x["step"], x["eval_loss"]) for x in logs if "eval_loss" in x]

plt.figure(figsize=(10, 5))
if train_loss:
    steps, losses = zip(*train_loss)
    plt.plot(steps, losses, label="Train Loss", alpha=0.7)
if eval_loss:
    steps, losses = zip(*eval_loss)
    plt.plot(steps, losses, label="Eval Loss", marker="o")

plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Qwen3-8B Recommend - Training Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("recommend_8b_training_curve.png", dpi=150)
plt.show()

## 10. [AFTER] 파인튜닝 후 테스트

In [ ]:
print("=" * 60)
print("[AFTER] 파인튜닝 후")
print("=" * 60)
after_result = run_inference(model, tokenizer, TEST_INPUT)
print(after_result[:800])

# JSON 파싱 확인
try:
    clean = after_result
    if "```json" in clean:
        clean = clean.split("```json")[1].split("```")[0].strip()
    elif "```" in clean:
        clean = clean.split("```")[1].split("```")[0].strip()
    parsed = json.loads(clean)
    print(f"\nJSON 파싱 성공!")
    print(f"필드: {list(parsed.keys())}")
except Exception as e:
    print(f"\nJSON 파싱 실패: {e}")

In [ ]:
# Before vs After 비교
print("=" * 60)
print("[BEFORE]")
print("-" * 40)
print(before_result[:500])
print()
print("[AFTER]")
print("-" * 40)
print(after_result[:500])
print("=" * 60)

## 11. Test 정량 평가 (JSON 유효율)

In [ ]:
required_keys = ["name", "intro", "description", "topic", "format",
                 "difficulty", "goal", "textbook", "processDetail",
                 "durationWeeks", "scheduleSuggestion"]

eval_count = min(30, len(val_raw))
json_valid = 0
key_complete = 0

for i in range(eval_count):
    text = val_raw[i]["text"]
    user_start = text.find("<|im_start|>user\n") + len("<|im_start|>user\n")
    user_end = text.find("<|im_end|>", user_start)
    user_msg = text[user_start:user_end]

    result = run_inference(model, tokenizer, user_msg)

    try:
        clean = result
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0].strip()
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0].strip()
        parsed = json.loads(clean)
        json_valid += 1
        missing = [k for k in required_keys if k not in parsed]
        if not missing:
            key_complete += 1
    except:
        pass

    if (i + 1) % 10 == 0:
        print(f"  진행: {i+1}/{eval_count} (JSON: {json_valid}, 키 완전: {key_complete})")

print(f"\n[결과]")
print(f"  JSON 유효율: {json_valid}/{eval_count} ({json_valid/eval_count:.1%})")
print(f"  키 완전율: {key_complete}/{eval_count} ({key_complete/eval_count:.1%})")

## 12. 모델 저장 + GGUF 변환

In [ ]:
LORA_SAVE = f"{OUTPUT_DIR}/lora"
model.save_pretrained(LORA_SAVE)
tokenizer.save_pretrained(LORA_SAVE)
print(f"LoRA 저장: {LORA_SAVE}")

In [ ]:
# GGUF 변환
GGUF_DIR = "./models/qwen3-8b-recommend"
print("GGUF Q4_K_M 변환 중...")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
print(f"GGUF 저장: {GGUF_DIR}")

for f in os.listdir(GGUF_DIR):
    if f.endswith(".gguf"):
        size = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1024**3
        print(f"  {f}: {size:.1f} GB")